# Day 20: Building a Program Function by Function

**Today's question:** you have a function plan. In what **order** do you build it?

We practise on a different game, **Rock, Paper, Scissors**, so that Hangman stays yours to build.

Run each cell with **Shift+Enter**. Before running a cell marked 🤔, predict what it will show.

## 1. The plan

Rules: the player and the computer each choose rock, paper or scissors. Rock beats scissors, scissors beats
paper, paper beats rock. Same choice: draw. We play 3 rounds and show the score.

| Function | Gets | Gives back | Check with `print(... == ...)`? |
|----------|------|------------|------|
| `decide_winner(player, computer)` | two choices | `"player"`, `"computer"` or `"draw"` | yes |
| `is_valid_choice(choice)` | text | `True`/`False` | yes |
| `get_computer_choice()` | nothing | a random choice | partly (random) |
| `get_player_choice()` | nothing | a valid choice | no: `input()` |
| `play_round()` | nothing | the winner | no: `input()` |
| `play_game(rounds=3)` | number of rounds | nothing (prints the score) | no |

**Build order:** checkable functions first (most logic, no `input()`), then the input functions, then the "glue"
functions that call everything else.

In [ ]:
import random

CHOICES = ["rock", "paper", "scissors"]
BEATS = {"rock": "scissors", "paper": "rock", "scissors": "paper"}  # each key beats its value

## 2. Step 1: `decide_winner`, checks first

Stub and checks before any logic:

In [ ]:
def decide_winner(player, computer):
    """Return "player", "computer" or "draw" for one round."""


print(decide_winner("rock", "scissors") == "player")
print(decide_winner("rock", "paper") == "computer")
print(decide_winner("paper", "paper") == "draw")
print(decide_winner("scissors", "paper") == "player")

A first attempt at the body:

In [ ]:
def decide_winner(player, computer):
    """Return "player", "computer" or "draw" for one round."""
    if BEATS[player] == computer:
        return "player"
    return "computer"

🤔 **Predict:** which check will still print `False`?

In [ ]:
print(decide_winner("rock", "scissors") == "player")
print(decide_winner("rock", "paper") == "computer")
print(decide_winner("paper", "paper") == "draw")
print(decide_winner("scissors", "paper") == "player")

The draw case was forgotten. The check pointed straight at it. Fix, then re-run **all** the checks:

In [ ]:
def decide_winner(player, computer):
    """Return "player", "computer" or "draw" for one round."""
    if player == computer:
        return "draw"
    if BEATS[player] == computer:
        return "player"
    return "computer"


print(decide_winner("rock", "scissors") == "player")
print(decide_winner("rock", "paper") == "computer")
print(decide_winner("paper", "paper") == "draw")
print(decide_winner("scissors", "paper") == "player")

## 3. Step 2: `is_valid_choice`

In [ ]:
def is_valid_choice(choice):
    """Return True if choice is exactly "rock", "paper" or "scissors"."""
    return choice in CHOICES


print(is_valid_choice("rock") == True)
print(is_valid_choice("lizard") == False)
print(is_valid_choice("Rock") == False)  # the caller must lowercase first

Why does validation matter? Without it, bad text reaches `decide_winner` and crashes the game:

In [ ]:
decide_winner("Rock", "paper")

## 4. Step 3: `get_computer_choice` (random: check what you can)

We cannot know the exact result of a random choice, but we can check that it is always **allowed**.

In [ ]:
def get_computer_choice():
    """Return a random choice for the computer."""
    return random.choice(CHOICES)


random.seed(4)
print(get_computer_choice() in CHOICES)
print(get_computer_choice() in CHOICES)

## 5. Step 4: `get_player_choice` (input: try it by hand)

Functions with `input()` are tried by running them and typing good **and** bad answers.
(In this notebook the "typed" answers are `lizard`, then ` Rock` with a space.)

In [ ]:
def get_player_choice():
    """Ask until the player types rock, paper or scissors; return it in lowercase."""
    while True:
        choice = input("rock, paper or scissors? ").strip().lower()
        if is_valid_choice(choice):
            return choice
        print("Please type rock, paper or scissors.")


print("You chose:", get_player_choice())

The rules live in `is_valid_choice` (checked). `get_player_choice` only asks, cleans and repeats.

## 6. Step 5: the glue functions

In [ ]:
def play_round():
    """Play one round and return the winner: "player", "computer" or "draw"."""
    player = get_player_choice()
    computer = get_computer_choice()
    print(f"You: {player} | Computer: {computer}")
    winner = decide_winner(player, computer)
    if winner == "draw":
        print("Draw!")
    elif winner == "player":
        print("You win this round!")
    else:
        print("The computer wins this round.")
    return winner

In [ ]:
def play_game(rounds=3):
    """Play several rounds and print the final score."""
    scores = {"player": 0, "computer": 0, "draw": 0}
    for round_number in range(1, rounds + 1):
        print(f"--- Round {round_number} ---")
        winner = play_round()
        scores[winner] += 1
    print(f"Final score: you {scores['player']}, computer {scores['computer']}, draws {scores['draw']}")

🤔 Because every piece is already checked or tried, the first full run usually just works:

In [ ]:
random.seed(7)
play_game()

## 7. From the notebook to a `.py` file

A game runs in the terminal, so it lives in a `.py` file. Same order as above:

```python
import random

CHOICES = [...]                 # constants at the top

def decide_winner(...): ...     # calculating functions
def get_player_choice(): ...    # input functions
def play_game(...): ...         # glue

def run_checks():
    """Print one True/False line per check."""
    print(decide_winner("rock", "scissors") == "player")
    ...

CHECK_ONLY = True               # True while building, False to play

if CHECK_ONLY:
    run_checks()
else:
    play_game()
```

The `CHECK_ONLY` switch lets you run the checks **without** starting the game. Your Hangman starter works
exactly like this.

## 8. Common mistakes on project day

| Mistake | Better |
|---------|--------|
| Writing the whole game, then testing by playing it again and again | Build and check the calculating functions first |
| A calculating function that prints its answer | `return` it, so other functions (and checks) can use it |
| "Play again" by calling `play_game()` inside `play_game()` | Use a `while` loop with a yes/no answer |
| Changing a function and not re-running its checks | Re-run **all** checks after every change |

## Recap

1. Plan first (Day 19), then build in this order: **checkable functions → input functions → glue**.
2. Write the checks before the body; a `False` points at the bug.
3. Random results: check that the result is **allowed** (`in` the list). Input functions: try them by hand with
   good and bad answers.
4. In a `.py` file, a `CHECK_ONLY` switch runs the checks without starting the game.

**Now:** build Hangman the same way.